# Image Folding
This use case applies when both sides of a GIWAXS pattern (i.e., $q_{xy}$ ranging from negative to positive values) have been measured. Since the current implementation of the `mlgidBASE` (0.1.2) package requires only positive values for the $q_{xy}$ and $q_{z}$ axes, the image can be folded after conversion but before being written to the NeXus file.

Additionally, this procedure can reduce the impact of detector gaps, as `numpy.nanmean` is used during averaging.

## Procedure

### 1. Perform the standard conversion routine
First, process the raw detector data using the standard conversion workflow to obtain the 2D reciprocal space map.

In [ ]:
import pygid
import numpy as np

exp_metadata = pygid.ExpMetadata(
    start_time = r"2018-05-17T04:42:59",
    end_time = r"2018-05-17T04:44:29",
    source_type = "synchrotron",
    source_name = "Elettra XRD1",
    detector_name = "pilatus2m",
    instrument_name = "XRD1",
)


data = {
    'name': 'KOLL_100mgNAB',
    'structure': {
        'stack': 'air | NAB | Koll:NAB | SiOx | Si',
        'materials': {
            'NAB': {
                'cas_number': '42924-53-8',
                'cif': 'NAB_form1.cif',
                'name': 'Nabumeton',
                'type': 'crystalline film'
              },
            'Koll': {
                'cas_number': '9003-39-8',
                'name': 'Kollidon',
                'type': 'amorphous film'
              },
            'SiOx': {
                'cas_number': '7631-86-9',
                'name': 'native SiOx',
              },
            'Si': {
                'cas_number': '7440-21-3',
                'name': 'Si wafer',
                'thickness': 0.0005
              },
          }

      },
        'experimental_conditions': 'standard conditions, in air',
        'preparation': 'spin coating from 100 mg/ml solution in CHCl3 at 30 rps',
}
smpl_metadata = pygid.SampleMetadata(path_to_save="KOLL_100mgNAB.yaml", data=data)

poni_path = './Elettra_2018_05_LaB6_calibration.poni'
mask_path = './Elettra202506_Pilatus2M_pixelmask_bigger.npy'
filename = './KOLL_100mgNAB_om52d4_1_sum.tiff'
save_fn = 'Nabumetone_Koll_100mg'

params = pygid.ExpParams(
	poni_path = poni_path,
    mask_path = mask_path,
    fliplr = True,
    flipud = True,
    transp = False,
    ai = 2.027
)

matrix = pygid.CoordMaps(
    params,
    vert_positive = True, hor_positive = False,  ### NOTE: DON'T CUT HERE
    q_xy_range = (-3.2, 3.2), q_z_range = (0, 3.5), dq = 0.002 ### NOTE: GIVE FULL q_xy RANGE.
)

analysis = pygid.Conversion(
    matrix = matrix,
    path = filename,
)

q_xy, q_z, img = analysis.det2q_gid(
    plot_result = False, return_result = True, ### NOTE: DON'T SAVE HERE, BUT RETURN THE ARRAYS
)

### 2. Fold the image along the vertical axis
Fold the data along the \( q_{xy} = 0 \) axis by combining the negative and positive halves of the pattern. Use appropriate averaging (e.g., `numpy.nanmean`) to handle missing values and detector gaps.


In [ ]:
img = np.asarray(img)
x = np.asarray(q_xy)
c = np.where(x > 0)[0][0]
n = min(img.shape[2] - c, c + 1)
img_sym = np.nanmean(
    np.stack([img[:, :, c:c+n], img[:, :, :c+1][:, :, ::-1][:, :, :n]]),
    axis=0
)
analysis.img_gid_q  = np.concatenate([img_sym, img[:, :, c+n:]], axis=2)
for matrix in analysis.matrix:
    matrix.q_xy = np.concatenate([x[c:c+n], x[c+n:]])

### 3. Save the result

In [ ]:
pygid.DataSaver(sample=analysis, path_to_save = f"{save_fn}.h5", overwrite_file=True,
                h5_group = "scan0", smpl_metadata = smpl_metadata, exp_metadata=exp_metadata)
